# Semantic Retrieval Pipeline using Docling Outputs

This notebook builds a simple semantic retrieval system using OCR-processed documents.

The goal is to test whether documents converted using Docling can be used in
downstream AI workflows such as RAG (Retrieval-Augmented Generation).

Documents used:
- Swahili travel phrases
- French reading sample

We move from raw text → structured chunks → embeddings → retrieval.

## 1) Objective
We want to answer one key question:

Can OCR-generated Markdown files be turned into a searchable knowledge base?

To test this, we:
1. Clean the text
2. Break it into smaller chunks
3. Convert chunks into embeddings
4. Retrieve the most relevant chunks based on a query

## 2) Methodology Overview 
Pipeline steps:

1. Load OCR outputs
2. Clean text slightly
3. Split text into chunks
4. Generate embeddings using SentenceTransformers
5. Perform similarity-based retrieval

This mimics the core idea behind RAG systems.

## 3) Installing Embedding Dependencies

Before running the retrieval pipeline, we need to install the `sentence-transformers` library.

This library is used to generate embeddings, which allow us to compare text based on meaning rather than exact words.

To ensure the package is installed in the same environment as the notebook, we use the Python executable linked to the current kernel:

```python
import sys
!{sys.executable} -m pip install sentence-transformers

In [1]:
import sys
!{sys.executable} -m pip install sentence-transformers

  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.9.0-py3-none-any.whl.metadata (14 kB)
  Using cached torch-2.11.0-cp314-cp314-win_amd64.whl.metadata (29 kB)
  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp314-cp314-win_amd64.whl.metadata (60 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

## 4) Imports
We import the tools needed for:
- embeddings (SentenceTransformer)
- numerical operations (numpy)
- text cleaning (re)

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
import re

## 5) Load Files 
We load the Markdown outputs generated earlier using Docling.

These are the same files we analyzed in the previous notebook.

In [4]:
with open("../03-outputs/Swahili-words-and-phrases-for-travelers.md", "r", encoding="utf-8") as f:
    swahili_text = f.read()

with open("../03-outputs/French-sample.md", "r", encoding="utf-8") as f:
    french_text = f.read()

print("Swahili length:", len(swahili_text))
print("French length:", len(french_text))

Swahili length: 229657
French length: 7813


## 6) Clean the text 
OCR output can sometimes include unnecessary spacing or formatting issues.

Here we lightly clean the text by:
- reducing excessive newlines
- trimming whitespace

This helps improve chunking and embeddings later.

In [5]:
def clean_text(text):
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()

swahili_text = clean_text(swahili_text)
french_text = clean_text(french_text)

## 7) Chuncking the text into smaller pieces

Instead of embedding the entire document at once, we split it into smaller chunks.

Why?
- Embeddings work better on smaller, meaningful pieces
- Retrieval becomes more precise

We split text into chunks of ~120 words.

In [ ]:
def chunk_text(text, chunk_size=120):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

swahili_chunks = chunk_text(swahili_text)
french_chunks = chunk_text(french_text)

print("Swahili chunks:", len(swahili_chunks))
print("French chunks:", len(french_chunks))

Swahili chunks: 15
French chunks: 12


## 8) Combine Chunks with Labels 
We store each chunk together with its source (Swahili or French).

This helps us track where retrieved results are coming from.

In [9]:
documents = []

for chunk in swahili_chunks:
    documents.append({"source": "Swahili", "text": chunk})

for chunk in french_chunks:
    documents.append({"source": "French", "text": chunk})

print("Total chunks:", len(documents))

Total chunks: 27


## 9) Load Embedding Model
We use a pre-trained embedding model from SentenceTransformers.

This model converts text into numerical vectors that capture meaning.

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7660.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 10) Create Emeddings 
We convert all text chunks into embeddings.

Each chunk becomes a vector representation that we can compare mathematically.

In [14]:
texts = [doc["text"] for doc in documents]
embeddings = model.encode(texts)

print("Embedding shape:", np.array(embeddings).shape)

Embedding shape: (27, 384)


## 11) retrieval Function
This function takes a query and returns the most relevant chunks.

Steps:
1. Convert query into embedding
2. Compute similarity with all document embeddings
3. Return top matches

In [15]:
def retrieve(query, documents, embeddings, model, top_k=3):
    query_embedding = model.encode([query])[0]
    scores = np.dot(embeddings, query_embedding)

    top_indices = np.argsort(scores)[-top_k:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            "source": documents[idx]["source"],
            "text": documents[idx]["text"],
            "score": float(scores[idx])
        })
    return results

## 12) Test Query 1
We test a query related to Swahili greetings.

In [16]:
results = retrieve("How do I greet someone in Swahili?", documents, embeddings, model)

for i, r in enumerate(results, 1):
    print(f"Result {i}")
    print("Source:", r["source"])
    print("Score:", round(r["score"], 4))
    print(r["text"][:500])
    print("-" * 80)

Result 1
Source: Swahili
Score: 0.4661
nawe Goodbye Kwaheri (See you) later! (Tutaonana) baadaye Good day (as goodbye) Siku njema Good evening (as goodbye) Jioni njema Good night (as goodbye) Usiku mwema Sleep well Lala salama ## COURTESIES &amp; (DIS)AGREEMENTS Yes Ndiyo Do you understand? Unaelewa? No Hapana I understand Naelewa OK Sawa I don't understand Sielewi Thank you Ahsante (to 1 person) I don't speak Swahili Siongei Kiswahili Ahsanteni (to more than 1 person) Could you repeat? Sema tena (You're) welcome Karibu (to 1 person)
--------------------------------------------------------------------------------
Result 2
Source: Swahili
Score: 0.4382
(6 + 6) minus a quarter in the evening). Please realize that Tanzanians are generally less strict about time than Europeans or Américains. This isn't considered rude, especially in situations that aren't work-related but even in those, it's normal. Our guides and staff members are of course aware that timings need to be respected when dea

## 13) Test Query 2
We test a query related to French reading concepts.

In [17]:
results = retrieve("What are cognates in French?", documents, embeddings, model)

for i, r in enumerate(results, 1):
    print(f"Result {i}")
    print("Source:", r["source"])
    print("Score:", round(r["score"], 4))
    print(r["text"][:500])
    print("-" * 80)

Result 1
Source: French
Score: 0.7374
complicated grammatical constructions and how they're used. ## II. Cognates and faux amis One of the best things about reading French is that there is a wealth of cognates. Cognates are words that share a common etymological origin. This can mean that they descended from a common parent language, or that they've been directly borrowed from another language. The technical definition of a cognate can be complicated, so for our purposes we'll rely on how the term is typically used in language learn
--------------------------------------------------------------------------------
Result 2
Source: French
Score: 0.6503
introduction , communication , passion , obsession , conclusion , etc. Others may have slight spelling differences but will otherwise be easy to decipher: dictionnaire , chocolat , dentiste , vanille , and famille , for example. There are many great resources on the Internet where you can find long lists of FrenchEnglish cognates, and you s

## 14) Supporting Setup 

Below is the setup step where we installed the embedding model dependency required for semantic search.

![Install Sentence Transformers](https://raw.githubusercontent.com/Uxer-Janine/outreachy-docling-task/master/05-screenshots/24-install-sentence-transformers.png)

## 15) Key Notes

This is not a full chatbot.

It is a retrieval system that:
- finds relevant information
- returns it without generating new text

Pipeline recap:
1. OCR (Docling)
2. Markdown output
3. Chunking
4. Embeddings
5. Retrieval

This is the foundation of RAG systems.

## Conclusion 

This experiment shows that OCR-processed documents can be used for semantic retrieval.

However:
- OCR noise affects embedding quality
- Cleaning improves results
- Chunking is essential for performance

Overall:
Docling outputs are usable for downstream AI workflows like RAG.
